In [28]:
!pip install ultralytics
!git clone https://github.com/iskabell/Recipe-Generator
!python /content/Recipe-Generator/Data/grocery_store/unpack_images.py

fatal: destination path 'Recipe-Generator' already exists and is not an empty directory.
Verified and extracted 5421 processed images


In [29]:
import pandas as pd
m = pd.read_csv("/content/Recipe-Generator/Data/grocery_store/processed/manifest.csv")
print(m.columns.tolist())
print(m["split"].value_counts())
m.head(3)

import shutil
from pathlib import Path

ROOT = Path("/content/Recipe-Generator/Data/grocery_store/processed")
OUT = Path("/content/dataset")
LABEL_COL = "ingredient"        # <- change to match the column name you see
IMAGE_COL = "image_path"

assert LABEL_COL in m.columns, f"Pick one of: {m.columns.tolist()}"
split_names = {"train": "train", "validation": "val", "val": "val", "test": "test"}

for _, r in m.iterrows():
    label = str(r[LABEL_COL]).strip().lower().replace(" ", "_").replace("/", "-")
    dest = OUT / split_names[r["split"]] / label
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(ROOT / r[IMAGE_COL], dest)

for s in ["train", "val", "test"]:
    n_cls = len(list((OUT / s).iterdir()))
    n_img = len(list((OUT / s).rglob("*.jpg")))
    print(s, "->", n_cls, "classes,", n_img, "images")

['image_path', 'source_path', 'split', 'fine_label', 'fine_name', 'label', 'coarse_name', 'ingredient', 'category', 'original_width', 'original_height', 'original_mode', 'source_sha256', 'pixel_sha256', 'processed_sha256']
split
train         2640
test          2485
validation     296
Name: count, dtype: int64
train -> 43 classes, 2640 images
val -> 37 classes, 296 images
test -> 43 classes, 2485 images


In [30]:
import random, shutil
from pathlib import Path

random.seed(42)
SRC = Path("/content/dataset")
DST = Path("/content/dataset_dev")
if DST.exists():
    shutil.rmtree(DST)

shutil.copytree(SRC / "test", DST / "test")

for cls_dir in sorted((SRC / "train").iterdir()):
    imgs = sorted(cls_dir.glob("*.jpg"))
    random.shuffle(imgs)
    n_val = max(1, round(0.1 * len(imgs))) if len(imgs) >= 2 else 0
    for i, p in enumerate(imgs):
        split = "val" if i < n_val else "train"
        (DST / split / cls_dir.name).mkdir(parents=True, exist_ok=True)
        shutil.copy(p, DST / split / cls_dir.name)

for s in ["train", "val", "test"]:
    print(s, "->", len(list((DST / s).iterdir())), "classes,",
          len(list((DST / s).rglob("*.jpg"))), "images")

train -> 43 classes, 2375 images
val -> 43 classes, 265 images
test -> 43 classes, 2485 images


In [31]:
from ultralytics import YOLO

model = YOLO("yolo26n-cls.pt")
model.train(data="/content/dataset_dev", epochs=20, imgsz=224, batch=32, seed=42,
            project="/content/runs", name="yolo26n_cls_20ep")

best = YOLO(model.trainer.best)
m_test = best.val(data="/content/dataset_dev", split="test")
print("YOLO26n-cls TEST  top1:", m_test.top1, " top5:", m_test.top5)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_dev, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_cls_20ep-2, nbs=64, nms=None, opset=None, optimize=False

In [32]:
!rm -rf /content/Recipe-Generator/Data/grocery_store/runs

In [33]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/recipe_runs
!cp -r /content/runs/yolo26n_cls_20ep-2 /content/drive/MyDrive/recipe_runs/

Mounted at /content/drive


In [34]:
import time, torch, torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
                               transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), norm])
eval_tf = transforms.Compose([transforms.ToTensor(), norm])

D = "/content/dataset_dev"
train_ds = datasets.ImageFolder(f"{D}/train", train_tf)
val_ds   = datasets.ImageFolder(f"{D}/val", eval_tf)
test_ds  = datasets.ImageFolder(f"{D}/test", eval_tf)
assert train_ds.classes == val_ds.classes == test_ds.classes, "class mismatch!"

loaders = {"train": DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2),
           "val":   DataLoader(val_ds, batch_size=32, num_workers=2),
           "test":  DataLoader(test_ds, batch_size=32, num_workers=2)}
print(len(train_ds.classes), "classes on", device)

43 classes on cuda


In [35]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.0005)
loss_fn = nn.CrossEntropyLoss()

def evaluate(loader):
    model.eval(); top1 = top5 = n = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).topk(5, dim=1).indices
            top1 += (pred[:, 0] == y).sum().item()
            top5 += (pred == y[:, None]).any(dim=1).sum().item()
            n += len(y)
    return top1 / n, top5 / n

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 179MB/s]


In [36]:
import os
os.makedirs("/content/runs/resnet18", exist_ok=True)
best, start = 0, time.time()
for epoch in range(20):
    model.train()
    for x, y in loaders["train"]:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    v1, _ = evaluate(loaders["val"])
    print(f"epoch {epoch+1:2d}: val top1 = {v1:.3f}")
    if v1 >= best:
        best = v1; torch.save(model.state_dict(), "/content/runs/resnet18/best.pt")
print(f"training time: {(time.time()-start)/60:.1f} min")

model.load_state_dict(torch.load("/content/runs/resnet18/best.pt"))
t1, t5 = evaluate(loaders["test"])
print(f"ResNet-18 TEST  top1: {t1:.4f}  top5: {t5:.4f}")
print("parameters:", sum(p.numel() for p in model.parameters()))

epoch  1: val top1 = 0.853
epoch  2: val top1 = 0.970
epoch  3: val top1 = 0.985
epoch  4: val top1 = 0.974
epoch  5: val top1 = 0.989
epoch  6: val top1 = 0.981
epoch  7: val top1 = 0.970
epoch  8: val top1 = 0.992
epoch  9: val top1 = 0.996
epoch 10: val top1 = 0.985
epoch 11: val top1 = 0.977
epoch 12: val top1 = 0.970
epoch 13: val top1 = 0.977
epoch 14: val top1 = 0.966
epoch 15: val top1 = 0.992
epoch 16: val top1 = 0.996
epoch 17: val top1 = 0.992
epoch 18: val top1 = 1.000
epoch 19: val top1 = 0.992
epoch 20: val top1 = 0.989
training time: 3.4 min
ResNet-18 TEST  top1: 0.8624  top5: 0.9855
parameters: 11198571


In [37]:
import os, time
size_mb = os.path.getsize("/content/runs/resnet18/best.pt") / 1e6
x = torch.randn(1, 3, 224, 224, device=device)
model.eval()
with torch.no_grad():
    for _ in range(20): model(x)                  # warm-up
    torch.cuda.synchronize(); t = time.time()
    for _ in range(200): model(x)
    torch.cuda.synchronize(); ms = (time.time() - t) / 200 * 1000
print(f"ResNet-18: {size_mb:.1f} MB, {ms:.1f} ms/image")

ResNet-18: 44.9 MB, 3.0 ms/image


In [38]:
from collections import Counter
from ultralytics import YOLO

yolo = YOLO("/content/runs/yolo26n_cls_20ep-2/weights/best.pt")
truth = [test_ds.classes[y] for _, y in test_ds.samples]
y_pred = [yolo.names[r.probs.top1]
          for r in yolo.predict([p for p, _ in test_ds.samples], imgsz=224, verbose=False, stream=True)]

model.eval(); r_pred = []
with torch.no_grad():
    for x, _ in loaders["test"]:
        r_pred += [test_ds.classes[i] for i in model(x.to(device)).argmax(1).tolist()]

print("YOLO worst classes:  ", Counter(t for t, p in zip(truth, y_pred) if t != p).most_common(8))
print("ResNet worst classes:", Counter(t for t, p in zip(truth, r_pred) if t != p).most_common(8))
both = sum(1 for t, a, b in zip(truth, y_pred, r_pred) if a != t and b != t)
print("wrong on the same images:", both)

YOLO worst classes:   [('satsumas', 30), ('orange', 27), ('apple', 20), ('melon', 19), ('lemon', 17), ('sour_cream', 17), ('pear', 16), ('red_grapefruit', 16)]
ResNet worst classes: [('satsumas', 49), ('melon', 30), ('lemon', 23), ('apple', 20), ('pear', 20), ('bell_pepper', 19), ('red_grapefruit', 18), ('orange', 17)]
wrong on the same images: 202
